# Imports

In [2]:
import numpy as np
import matplotlib as plt
from pathlib import Path

# Function

In [44]:
def COBRA_PWR_input_generator(

        N_v_slices=18,

        length=3.66,
        rod_pitch=0.0126,
        rod_diameter=0.0095,

        N_assemblies=193,
        N_holes=17*17,
        N_rods=264,

        plant_thermal_power=3320e6,
        mass_flow_rate=19000,

        pellet_diameter=0.008,
        cladding_thickness=0.00057,
        fuel_density=10000,
        clad_density=6500,
        fuel_conductivity=3.3,
        fuel_shc=300,
        clad_conductivity=10,
        clad_shc=500,
        fuel_clad_heat_trans_coeff=9400,
        inlet_temperature=290,
        system_pressure=15.5,

        filename="COBRA_input.txt"

):

    ####################
    # Calculate Variables
    ####################

    z_coordinates = [0.13 + 0.2*z for z in range(N_v_slices)]

    z_coordinates = np.array(z_coordinates)

    ###################################################
    # Variable 1: Power distribution as a function of z
    ###################################################

    q_max = (np.pi/(2*length))*(
        plant_thermal_power/(N_assemblies)
    )

    power_values = q_max*np.sin(
        (np.pi*z_coordinates)/length
    )

    ###############################
    # Variable 2: Channel Flow Area
    ###############################

    assembly_area = N_holes*rod_pitch**2

    hole_area = (
        N_holes*np.pi*(rod_diameter/2)**2
    )

    channel_flow_area = assembly_area-hole_area

    ######################################
    # Variable 3: Channel Wetted Perimeter
    ######################################

    channel_wetted_perimeter = (
        N_holes*np.pi*rod_diameter
    )

    ######################################
    # Variable 4: Channel Heated Perimeter
    ######################################

    channel_heated_perimeter = (
        N_rods*np.pi*rod_diameter
    )

    #########################################
    # Variables 5-14: Fuel and Cladding Properties
    #########################################

    # These are already defined as input parameters.

    #########################################
    # Variable 15: Inlet Temperature
    #########################################

    inlet_temperature_K = inlet_temperature + 273.15

    #################################################
    # Variable 16: Inlet Mass Flow Rate per Unit Area
    #################################################

    inlet_mass_flow_rate = (
        (mass_flow_rate/N_assemblies)/channel_flow_area
    )

    ##############################
    # Variable 17: System Pressure
    ##############################

    system_pressure_MPa = system_pressure


    ##############################
    # Format Variables
    ##############################

    # Variable 1: Linear fission power
    # Format: X.XXXXE+0X

    power_formatted = [
        f"{value:.4E}" for value in power_values
    ]

    # Variable 2: Channel flow area
    # Format: X.XXXE-0X

    flow_area_formatted = f"{channel_flow_area:.3E}"

    # Variables 3 and 4: Perimeters
    # Format: X.XXXXX

    wetted_perimeter_formatted = (
        f"{channel_wetted_perimeter:.5f}"
    )

    heated_perimeter_formatted = (
        f"{channel_heated_perimeter:.5f}"
    )

    # Number of fuel rods
    # Format: XXX

    rods_formatted = f"{N_rods:03d}"

    # Variable 5: Pellet diameter
    # Format: X.XXX

    pellet_formatted = f"{pellet_diameter:.3f}"

    # Variable 6: Cladding thickness
    # Format: X.XXXXX

    clad_thickness_formatted = (
        f"{cladding_thickness:.5f}"
    )

    # Variable 7: Fuel density
    # Format: XXXXX

    fuel_density_formatted = f"{fuel_density:05d}"

    # Variable 8: Clad density
    # Format: XXXX

    clad_density_formatted = f"{clad_density:04d}"

    # Variable 9: Rod diameter
    # Format: X.XXXX

    rod_diameter_formatted = f"{rod_diameter:.4f}"

    # Variable 10: Fuel conductivity
    # Format: X.X

    fuel_conductivity_formatted = (
        f"{fuel_conductivity:.1f}"
    )

    # Variable 11: Fuel specific heat capacity
    # Format: XXX

    fuel_shc_formatted = f"{fuel_shc:03d}"

    # Variable 12: Clad conductivity
    # Format: XX

    clad_conductivity_formatted = (
        f"{clad_conductivity:02d}"
    )

    # Variable 13: Clad specific heat capacity
    # Format: XXX

    clad_shc_formatted = f"{clad_shc:03d}"

    # Variable 14: Fuel-clad heat transfer coefficient
    # Format: XXXX

    heat_trans_coeff_formatted = (
        f"{fuel_clad_heat_trans_coeff:04d}"
    )

    # Variable 15: Inlet temperature
    # Format: XXX

    inlet_temperature_formatted = (
        f"{round(inlet_temperature_K):03d}"
    )

    # Variable 16: Inlet mass flow rate
    # Format: XXXX

    inlet_mass_flow_formatted = (
        f"{round(inlet_mass_flow_rate):04d}"
    )

    # Variable 17: System pressure
    # Format: XX.X

    system_pressure_formatted = (
        f"{system_pressure_MPa:.1f}"
    )


    ##############################
    # Assemble COBRA Input File
    ##############################

    lines = []

    #initial empty line
    lines.append("")

    # Header
    lines.append("     0     2     2     1")

    lines.append(
        f"     2     1     0    {N_v_slices:2d}"
        "     1     0     0     5     0     0     0     1"
    )

    lines.append("   -0.20")
    lines.append(f"    {N_v_slices}")

    # Axial coordinates and linear fission power

    for z, power in zip(z_coordinates, power_formatted):

        lines.append(f"  {z:.2f}")
        lines.append(f"{power}")

    # Channel geometry: Variables 2, 3, 4 and number of rods

    lines.append(
        f"     1  1.0   {flow_area_formatted}    {wetted_perimeter_formatted}     {heated_perimeter_formatted}      {rods_formatted}        1       "
    )



    # Fuel and cladding geometry: Variables 5-9

    lines.append(
          f"  {pellet_formatted}       {clad_thickness_formatted}      {fuel_density_formatted}       {clad_density_formatted}        {rod_diameter_formatted}  "
    )



    ################
    #Correct to here
    ################



    # Thermal properties: Variables 10-14

    lines.append(
        f"    {fuel_conductivity_formatted}       {fuel_shc_formatted}          {clad_conductivity_formatted}          {clad_shc_formatted}         {heat_trans_coeff_formatted} "
    )

    # Remaining fixed COBRA records

    lines.append(
        "     0     0     1     1     0     1     1     0     1     0"
    )

    lines.append("     0")
    lines.append("$            JSLIP")
    lines.append("     0     0     0")
    lines.append("     0     0     0     0     0     0")
    lines.append(" 10.   0.0   0.5  0.0")
    lines.append(
        "     3     0     0  0.0      0     0    0.    0.    0."
    )




    lines.append(
        "$                  WERRX DAMPP DAMPF                         EPSP ISCHEM"
    )

    lines.append(
        "     0     0    0.  0.     0.    0.  -.01     0.     0  0.     0.      2"
    )

    # Operating conditions: Variables 15, 16 and 17

    lines.append(
        f"     1  {inlet_temperature_formatted}  {inlet_mass_flow_formatted}  {system_pressure_formatted}    -20     1   0.  0.0   0.0  1600.   0.0  0.0"
    )

    # Final records

    lines.append("     0     0     0")
    lines.append("     0     0     0     0     0")
    lines.append("     0     5     3     0     0     0    -1")
    lines.append("$EOD")


    ##############################
    # Write .txt File
    ##############################

    with open(filename, "w") as file:

        file.write("\n".join(lines))
        file.write("\n")


    ##############################
    # Return Results
    ##############################

    return {
        "filename": filename,
        "z_coordinates": z_coordinates,
        "power_values": power_values,
        "q_max": q_max,
        "channel_flow_area": channel_flow_area,
        "channel_wetted_perimeter": channel_wetted_perimeter,
        "channel_heated_perimeter": channel_heated_perimeter,
        "inlet_mass_flow_rate": inlet_mass_flow_rate,
        "system_pressure_MPa": system_pressure_MPa
    }


In [46]:
COBRA_PWR_input_generator(filename= "PWR_INP.txt")

{'filename': 'PWR_INP.txt',
 'z_coordinates': array([0.13, 0.33, 0.53, 0.73, 0.93, 1.13, 1.33, 1.53, 1.73, 1.93, 2.13,
        2.33, 2.53, 2.73, 2.93, 3.13, 3.33, 3.53]),
 'power_values': array([ 822110.23778762, 2063379.24149599, 3243987.21757404,
        4329225.62014852, 5287189.66148086, 6089716.27682202,
        6713212.08682213, 7139347.01633204, 7355593.17798458,
        7355593.17798458, 7139347.01633204, 6713212.08682213,
        6089716.27682202, 5287189.66148086, 4329225.62014851,
        3243987.21757405, 2063379.24149599,  822110.23778762]),
 'q_max': 7382773.868114975,
 'channel_flow_area': 0.025396688752726906,
 'channel_wetted_perimeter': 8.625242630430776,
 'channel_heated_perimeter': 7.879114375203201,
 'inlet_mass_flow_rate': 3876.3161927694973,
 'system_pressure_MPa': 15.5}